In [16]:
import duckdb
import pandas as pd
import numpy as np

DB_PATH = "developer_project.duckdb"
con = duckdb.connect(DB_PATH)

HMM_WEEKLY = "dev_hmm_weekly_states_v1"
HMM_STATE_PROFILE = "dev_hmm_state_profile_v1"
HDBSCAN_FINAL = "dev_lifecycle_cluster_membership_v11_final"
HMM_JOURNEY = "dev_hmm_developer_journey_v1"


In [17]:
# ============================================================
# 1. Inspect HMM state profile
# ============================================================

print("HMM state profile schema:")
display(con.execute(f"DESCRIBE {HMM_STATE_PROFILE}").df())

state_profile = con.execute(f"""
SELECT *
FROM {HMM_STATE_PROFILE}
ORDER BY hmm_hidden_state
""").df()

display(state_profile)


HMM state profile schema:


,column_name,column_type,null,key,default,extra
0,hmm_hidden_state,BIGINT,YES,None,None,None
1,suggested_state_label,VARCHAR,YES,None,None,None
2,n_weekly_rows,BIGINT,YES,None,None,None
3,avg_state_probability,DOUBLE,YES,None,None,None
4,top_weekly_gmm_clusters,VARCHAR,YES,None,None,None
5,top_hdbscan_strata,VARCHAR,YES,None,None,None
6,top_hdbscan_clusters,VARCHAR,YES,None,None,None


,hmm_hidden_state,suggested_state_label,n_weekly_rows,avg_state_probability,top_weekly_gmm_clusters,top_hdbscan_strata,top_hdbscan_clusters
0,0,RENAME_STATE_0,33037,0.614233,0 (100.0%),"dormant (65.6%), at_risk (25.3%), cooling (5.0...","Dormant_Low_Depth (42.6%), Dormant_Former_Buil..."
1,1,RENAME_STATE_1,580588,0.786148,"0 (96.6%), 2 (3.2%), 1 (0.2%)","dormant (50.3%), at_risk (27.2%), active (14.1...","Dormant_Former_Builders (27.8%), Dormant_Low_D..."
2,2,RENAME_STATE_2,81068,0.663508,"0 (65.1%), 2 (32.8%), 1 (2.1%)","dormant (52.0%), at_risk (28.9%), active (10.0...","Dormant_Low_Depth (29.8%), Dormant_Former_Buil..."
3,3,RENAME_STATE_3,35153,0.700400,"0 (78.7%), 1 (16.6%), 2 (4.8%)","dormant (44.5%), at_risk (28.9%), active (15.0...","Dormant_Former_Builders (31.1%), active_noise ..."
4,4,RENAME_STATE_4,2918,0.827679,"1 (58.5%), 0 (26.3%), 2 (15.2%)","dormant (58.8%), at_risk (23.5%), active (9.7%...","Dormant_Low_Depth (30.4%), Dormant_Former_Buil..."
5,5,RENAME_STATE_5,10096,0.624510,"1 (91.2%), 0 (6.9%), 2 (1.9%)","dormant (44.8%), at_risk (24.6%), active (20.5...","Dormant_Low_Depth (24.6%), active_noise (20.4%..."
6,6,RENAME_STATE_6,162940,0.828413,"2 (81.5%), 0 (15.4%), 1 (3.0%)","dormant (62.8%), at_risk (23.7%), active (7.9%...","Dormant_Low_Depth (33.9%), Dormant_Former_Buil..."
7,7,RENAME_STATE_7,10468,0.620363,"1 (92.0%), 0 (6.3%), 2 (1.7%)","dormant (45.2%), at_risk (24.2%), active (20.3...","Dormant_Low_Depth (25.5%), active_noise (20.3%..."


In [18]:
# ============================================================
# 2. Create / update manual HMM state labels
# Update these labels after inspecting state_profile
# ============================================================

HMM_STATE_LABELS = {
    0: "Low / Idle Activity",
    1: "Low Recent Activity / Lapsing",
    2: "At-Risk Engagement",
    3: "Cooling Engagement",
    4: "Active Exploration",
    5: "Build-Oriented Usage",
    6: "High-Engagement Power Usage",
    7: "Irregular / Noisy Activity",
}

label_df = pd.DataFrame([
    {
        "hmm_hidden_state": state,
        "hmm_state_label": label
    }
    for state, label in HMM_STATE_LABELS.items()
])

con.register("label_df_view", label_df)

con.execute("""
CREATE OR REPLACE TABLE dev_hmm_state_labels_v1 AS
SELECT *
FROM label_df_view
""")

con.unregister("label_df_view")

print("Saved: dev_hmm_state_labels_v1")
display(label_df)


Saved: dev_hmm_state_labels_v1


,hmm_hidden_state,hmm_state_label
0,0,Low / Idle Activity
1,1,Low Recent Activity / Lapsing
2,2,At-Risk Engagement
3,3,Cooling Engagement
4,4,Active Exploration
5,5,Build-Oriented Usage
6,6,High-Engagement Power Usage
7,7,Irregular / Noisy Activity


In [19]:
# ============================================================
# 3. Create HMM latest-state to HDBSCAN alignment
# ============================================================

hmm_hdbscan = con.execute(f"""
WITH latest_hmm AS (
    SELECT
        developer_id,
        week_start,
        hmm_hidden_state,
        hmm_state_probability,
        ROW_NUMBER() OVER (
            PARTITION BY developer_id
            ORDER BY week_start DESC
        ) AS rn
    FROM {HMM_WEEKLY}
)

SELECT
    h.developer_id,
    h.week_start AS latest_hmm_week_start,
    h.hmm_hidden_state,
    l.hmm_state_label,
    h.hmm_state_probability,
    c.stratum,
    c.cluster_key,
    c.hdbscan_cluster,
    c.cluster_probability,
    c.outlier_score,
    c.adoption_direction

FROM latest_hmm h

LEFT JOIN dev_hmm_state_labels_v1 l
    ON h.hmm_hidden_state = l.hmm_hidden_state

LEFT JOIN {HDBSCAN_FINAL} c
    ON h.developer_id = c.developer_id

WHERE h.rn = 1
""").df()

display(hmm_hdbscan.head())

con.register("hmm_hdbscan_view", hmm_hdbscan)

con.execute("""
CREATE OR REPLACE TABLE dev_hmm_hdbscan_alignment_v1 AS
SELECT *
FROM hmm_hdbscan_view
""")

con.unregister("hmm_hdbscan_view")

print("Saved: dev_hmm_hdbscan_alignment_v1")


,developer_id,latest_hmm_week_start,hmm_hidden_state,hmm_state_label,hmm_state_probability,stratum,cluster_key,hdbscan_cluster,cluster_probability,outlier_score,adoption_direction
0,4574743,2026-02-23,2,At-Risk Engagement,0.508335,active,active_noise,-1,0.0,NaN,accelerating_or_active
1,2773121,2026-03-02,1,Low Recent Activity / Lapsing,0.954688,active,active_noise,-1,0.0,NaN,accelerating_or_active
2,3082641,2026-03-09,1,Low Recent Activity / Lapsing,0.830407,active,active_noise,-1,0.0,NaN,accelerating_or_active
3,9553482,2026-02-23,1,Low Recent Activity / Lapsing,0.830407,active,active_noise,-1,0.0,NaN,accelerating_or_active
4,9154888,2026-02-16,1,Low Recent Activity / Lapsing,0.676028,active,active_noise,-1,0.0,NaN,accelerating_or_active


Saved: dev_hmm_hdbscan_alignment_v1


In [20]:
alignment_summary = con.execute("""
SELECT
    stratum,
    cluster_key,
    hmm_hidden_state,
    hmm_state_label,
    COUNT(*) AS n_developers,
    AVG(hmm_state_probability) AS avg_hmm_state_probability,
    AVG(cluster_probability) AS avg_hdbscan_cluster_probability,
    AVG(outlier_score) AS avg_hdbscan_outlier_score
FROM dev_hmm_hdbscan_alignment_v1
GROUP BY
    stratum,
    cluster_key,
    hmm_hidden_state,
    hmm_state_label
ORDER BY
    stratum,
    cluster_key,
    n_developers DESC
""").df()

display(alignment_summary)

con.register("alignment_summary_view", alignment_summary)

con.execute("""
CREATE OR REPLACE TABLE dev_hmm_hdbscan_alignment_summary_v1 AS
SELECT *
FROM alignment_summary_view
""")

con.unregister("alignment_summary_view")

print("Saved: dev_hmm_hdbscan_alignment_summary_v1")

,stratum,cluster_key,hmm_hidden_state,hmm_state_label,n_developers,avg_hmm_state_probability,avg_hdbscan_cluster_probability,avg_hdbscan_outlier_score
0,active,active_0,6,High-Engagement Power Usage,6,0.464742,0.876742,0.123258
1,active,active_0,1,Low Recent Activity / Lapsing,6,0.718556,0.793165,0.206835
2,active,active_1,1,Low Recent Activity / Lapsing,2144,0.763138,0.867727,0.132273
3,active,active_1,6,High-Engagement Power Usage,4,0.486790,0.327640,0.672360
4,active,active_1,2,At-Risk Engagement,1,0.536485,0.120724,0.879276
...,...,...,...,...,...,...,...,...
66,dormant,Dormant_Low_Depth,6,High-Engagement Power Usage,3886,0.443034,1.000000,0.000000
67,dormant,Dormant_Low_Depth,2,At-Risk Engagement,3572,0.601635,1.000000,0.000000
68,dormant,Dormant_Low_Depth,3,Cooling Engagement,896,0.576985,1.000000,0.000000
69,dormant,Dormant_Low_Depth,5,Build-Oriented Usage,489,0.601238,1.000000,0.000000


Saved: dev_hmm_hdbscan_alignment_summary_v1


In [21]:
# ============================================================
# 5. Simple mismatch / signal table
# HDBSCAN = long-term cohort
# HMM = latest weekly behavior
# ============================================================

mismatch = con.execute("""
SELECT
    stratum,
    hmm_hidden_state,
    hmm_state_label,
    COUNT(*) AS n_developers,
    AVG(hmm_state_probability) AS avg_hmm_state_probability
FROM dev_hmm_hdbscan_alignment_v1
GROUP BY
    stratum,
    hmm_hidden_state,
    hmm_state_label
ORDER BY
    stratum,
    n_developers DESC
""").df()

display(mismatch)

con.register("mismatch_view", mismatch)

con.execute("""
CREATE OR REPLACE TABLE dev_hmm_lifecycle_signal_summary_v1 AS
SELECT *
FROM mismatch_view
""")

con.unregister("mismatch_view")

print("Saved: dev_hmm_lifecycle_signal_summary_v1")


,stratum,hmm_hidden_state,hmm_state_label,n_developers,avg_hmm_state_probability
0,active,1,Low Recent Activity / Lapsing,11948,0.810056
1,active,2,At-Risk Engagement,328,0.587781
2,active,3,Cooling Engagement,312,0.651838
3,active,5,Build-Oriented Usage,140,0.615814
4,active,6,High-Engagement Power Usage,101,0.434812
5,active,7,Irregular / Noisy Activity,76,0.577009
6,at_risk,1,Low Recent Activity / Lapsing,31559,0.758246
7,at_risk,2,At-Risk Engagement,2149,0.604441
8,at_risk,6,High-Engagement Power Usage,1412,0.449212
9,at_risk,3,Cooling Engagement,973,0.616862


Saved: dev_hmm_lifecycle_signal_summary_v1


In [22]:
# ============================================================
# 6. Journey archetype summary
# ============================================================

journeys = con.execute(f"""
SELECT *
FROM {HMM_JOURNEY}
""").df()

display(journeys.head())

journey_summary = con.execute(f"""
SELECT
    first_hmm_state,
    last_hmm_state,
    dominant_hmm_state,
    stratum,
    cluster_key,
    COUNT(*) AS n_developers,
    AVG(avg_state_probability) AS avg_state_probability,
    AVG(hmm_state_entropy) AS avg_hmm_state_entropy
FROM {HMM_JOURNEY}
GROUP BY
    first_hmm_state,
    last_hmm_state,
    dominant_hmm_state,
    stratum,
    cluster_key
ORDER BY n_developers DESC
""").df()

display(journey_summary.head(50))

con.register("journey_summary_view", journey_summary)

con.execute("""
CREATE OR REPLACE TABLE dev_hmm_journey_archetype_summary_v1 AS
SELECT *
FROM journey_summary_view
""")

con.unregister("journey_summary_view")

print("Saved: dev_hmm_journey_archetype_summary_v1")

,developer_id,n_weeks,first_week,last_week,first_hmm_state,last_hmm_state,avg_state_probability,dominant_hmm_state,hmm_state_entropy,first_gmm_weekly_cluster,last_gmm_weekly_cluster,stratum,cluster_key,hdbscan_cluster,cluster_probability,outlier_score,adoption_direction
0,047e118b82c918a12e8ff838f6275c0d454cc39c3b6b89...,52,2023-10-23,2025-05-12,0,1,0.667409,2,1.243319,0,0,at_risk,at_risk_5,5,0.691785,0.308215,at_risk
1,1,8,2021-08-02,2024-12-23,6,1,0.844066,1,0.562335,2,0,dormant,Dormant_Former_Builders,0,1.000000,0.000000,steady_inactive
2,10000003,4,2026-01-05,2026-02-09,6,1,0.863110,1,0.562335,2,0,active,active_3,3,1.000000,0.000000,accelerating_or_active
3,10000024,8,2026-01-05,2026-03-09,6,1,0.926372,1,0.376770,2,0,active,active_3,3,1.000000,0.000000,accelerating_or_active
4,10000027,9,2026-01-05,2026-03-09,6,1,0.936654,1,0.348832,2,0,active,active_3,3,1.000000,0.000000,accelerating_or_active


,first_hmm_state,last_hmm_state,dominant_hmm_state,stratum,cluster_key,n_developers,avg_state_probability,avg_hmm_state_entropy
0,6,1,1,dormant,Dormant_Former_Builders,33397,0.794055,0.540210
1,6,1,1,dormant,Dormant_Low_Depth,31563,0.757921,0.565013
2,6,1,1,at_risk,at_risk_5,14747,0.778526,0.540200
3,6,1,1,at_risk,at_risk_0,7227,0.791881,0.522532
4,6,1,1,active,active_noise,5914,0.811469,0.457167
5,6,1,1,at_risk,at_risk_noise,5057,0.768874,0.468652
6,6,1,1,cooling,cooling_noise,5054,0.801187,0.468699
7,6,1,1,active,active_3,3060,0.870389,0.456882
8,0,1,0,dormant,Dormant_Low_Depth,2719,0.643151,1.074627
9,6,1,6,dormant,Dormant_Low_Depth,2590,0.733513,0.637909


Saved: dev_hmm_journey_archetype_summary_v1


In [23]:
con.close()